In [ ]:

!pip install yfinance pandas numpy matplotlib scikit-learn


In [ ]:

import yfinance as yf
import pandas as pd
import numpy as np

tickers = [
    "AAPL", "MSFT", "AMZN", "GOOGL", "NVDA",
    "JPM", "KO", "WMT", "XOM", "JNJ"
]

data_list = []

for ticker in tickers:
    df = yf.download(
        ticker,
        start="2015-01-01",
        end="2026-10-15",
        auto_adjust=True,
        progress=False
    )

    if df.empty:
        print(f"Нет данных: {ticker}")
        continue

    # Оставляем нужные столбцы
    df = df[["Close", "Volume"]].copy()

    # Убираем лишний уровень названий столбцов,
    # если он присутствует
    df.columns = ["Close", "Volume"]

    df["Ticker"] = ticker
    df.index.name = "Date"
    df = df.reset_index()

    data_list.append(df)

prices = pd.concat(data_list, ignore_index=True)
prices = prices.sort_values(["Date", "Ticker"])
prices = prices.reset_index(drop=True)

print("Размер таблицы:", prices.shape)
display(prices.head(10))


Размер таблицы: (29630, 4)


,Date,Close,Volume,Ticker
0,2015-01-02,24.171757,212818400,AAPL
1,2015-01-02,15.426000,55664000,AMZN
2,2015-01-02,26.227926,26480000,GOOGL
3,2015-01-02,75.737358,5753600,JNJ
4,2015-01-02,45.838100,12600000,JPM
5,2015-01-02,29.215931,9921100,KO
6,2015-01-02,39.607182,27913900,MSFT
7,2015-01-02,0.481884,113680000,NVDA
8,2015-01-02,23.072304,13505400,WMT
9,2015-01-02,56.777901,10220400,XOM


In [ ]:

# Типы данных
print("Типы столбцов:")
print(prices.dtypes)

# Пропущенные значения
print("\nКоличество пропусков:")
print(prices.isna().sum())

# Количество наблюдений по каждой акции
print("\nНаблюдения по акциям:")
print(prices.groupby("Ticker").size())

# Даты начала и конца
print("\nПериод данных:")
print("Начало:", prices["Date"].min())
print("Конец:", prices["Date"].max())

# Проверка дубликатов
print("\nДубликаты даты и тикера:",
      prices.duplicated(["Date", "Ticker"]).sum())


Типы столбцов:
Date      datetime64[ns]
Close            float64
Volume             int64
Ticker            object
dtype: object

Количество пропусков:
Date      0
Close     0
Volume    0
Ticker    0
dtype: int64

Наблюдения по акциям:
Ticker
AAPL     2963
AMZN     2963
GOOGL    2963
JNJ      2963
JPM      2963
KO       2963
MSFT     2963
NVDA     2963
WMT      2963
XOM      2963
dtype: int64

Период данных:
Начало: 2015-01-02 00:00:00
Конец: 2026-10-14 00:00:00

Дубликаты даты и тикера: 0


In [ ]:

prices = prices.sort_values(["Ticker", "Date"]).copy()

# Расчёты отдельно для каждой акции
groups = prices.groupby("Ticker", group_keys=False)

# Доходность на следующие 5 торговых дней
prices["future_return_5d"] = (
    groups["Close"].shift(-5) / prices["Close"] - 1
)

# Целевая переменная:
# 1 — цена выросла, 0 — не выросла
prices["target"] = np.where(
    prices["future_return_5d"].notna(),
    (prices["future_return_5d"] > 0).astype(int),
    np.nan
)

# Признаки должны использовать только прошлую информацию
g = prices.groupby("Ticker")["Close"]

prices["ret_1"] = g.transform(
    lambda s: s.pct_change().shift(1)
)

prices["ret_5"] = g.transform(
    lambda s: s.pct_change(5).shift(1)
)

prices["ret_20"] = g.transform(
    lambda s: s.pct_change(20).shift(1)
)

prices["momentum_60"] = g.transform(
    lambda s: s.pct_change(60).shift(1)
)

prices["volatility_5"] = g.transform(
    lambda s: s.pct_change().rolling(5).std().shift(1)
)

prices["volatility_20"] = g.transform(
    lambda s: s.pct_change().rolling(20).std().shift(1)
)

prices["ma_ratio_20"] = g.transform(
    lambda s: (s / s.rolling(20).mean() - 1).shift(1)
)

prices["ma_ratio_60"] = g.transform(
    lambda s: (s / s.rolling(60).mean() - 1).shift(1)
)

prices["volume_change"] = prices.groupby("Ticker")["Volume"].transform(
    lambda s: s.pct_change().shift(1)
)

prices["relative_volume_20"] = prices.groupby("Ticker")["Volume"].transform(
    lambda s: (s / s.rolling(20).mean()).shift(1)
)



In [ ]:
feature_info = {
    "ret_1":              "Доходность за 1 день — краткосрочное движение цены",
    "ret_5":              "Доходность за 5 дней — недельный импульс",
    "ret_20":             "Доходность за 20 дней — месячная динамика",
    "momentum_60":        "Доходность за 60 дней — среднесрочный тренд (моментум)",
    "volatility_5":       "Ст. отклонение доходности за 5 дней — краткосрочный риск",
    "volatility_20":      "Ст. отклонение доходности за 20 дней — месячный риск",
    "ma_ratio_20":        "Отклонение цены от средней за 20 дней — перекупленность/перепроданность",
    "ma_ratio_60":        "Отклонение цены от средней за 60 дней — положение относительно тренда",
    "volume_change":      "Изменение объёма торгов за день — всплеск активности",
    "relative_volume_20": "Объём относительно среднего за 20 дней — аномальная активность",
}

FEATURES = list(feature_info.keys())

desc = pd.DataFrame({
    "Тип": prices[FEATURES].dtypes.astype(str),
    "Доля пропусков, %": (prices[FEATURES].isna().mean() * 100).round(2),
    "Экономический смысл": pd.Series(feature_info),
})
display(desc)

# Проверка на бесконечные значения (например, деление на нулевой объём)
print("Бесконечные значения:", np.isinf(prices[FEATURES]).sum().sum())

,Тип,"Доля пропусков, %",Экономический смысл
ret_1,float64,0.07,Доходность за 1 день — краткосрочное движение ...
ret_5,float64,0.20,Доходность за 5 дней — недельный импульс
ret_20,float64,0.71,Доходность за 20 дней — месячная динамика
momentum_60,float64,2.06,Доходность за 60 дней — среднесрочный тренд (м...
volatility_5,float64,0.20,Ст. отклонение доходности за 5 дней — краткоср...
volatility_20,float64,0.71,Ст. отклонение доходности за 20 дней — месячны...
ma_ratio_20,float64,0.67,Отклонение цены от средней за 20 дней — переку...
ma_ratio_60,float64,2.02,Отклонение цены от средней за 60 дней — положе...
volume_change,float64,0.07,Изменение объёма торгов за день — всплеск акти...
relative_volume_20,float64,0.67,Объём относительно среднего за 20 дней — анома...


Бесконечные значения: 0


In [ ]:
# ---- Проверка утечки ----
# future_return_5d рассчитан по будущей цене -> это и есть ответ, в модель его подавать нельзя.
# Close и Volume за текущий день не используем как признаки: признаки строятся только по данным до дня t.
LEAKY_COLS = ["future_return_5d"]
print("Удаляем из признаков (утечка):", LEAKY_COLS)

# Проверка: ни один признак не должен сильно коррелировать с будущей доходностью
corr = prices[FEATURES + ["future_return_5d"]].corr()["future_return_5d"].drop("future_return_5d")
print("\nКорреляция признаков с будущей доходностью (должна быть близка к 0):")
print(corr.round(3))

# ---- Подготовка данных ----
model_data = prices.dropna(subset=["target"] + FEATURES).copy()
model_data["target"] = model_data["target"].astype(int)
model_data = model_data[["Date", "Ticker"] + FEATURES + ["target"]]

# ---- Разбиение по времени с зазором в 5 дней ----
HORIZON = 5  # горизонт прогноза
dates = np.sort(model_data["Date"].unique())
n_dates = len(dates)
train_end = int(n_dates * 0.70)
valid_end = int(n_dates * 0.85)

# Зазор: target последних 5 дней train «смотрит» в период validation -> выкидываем их
train_dates = dates[:train_end - HORIZON]
valid_dates = dates[train_end:valid_end - HORIZON]
test_dates  = dates[valid_end:]

train = model_data[model_data["Date"].isin(train_dates)].copy()
valid = model_data[model_data["Date"].isin(valid_dates)].copy()
test  = model_data[model_data["Date"].isin(test_dates)].copy()

for name, d in [("Train", train), ("Validation", valid), ("Test", test)]:
    print(f"{name:11s}: {d['Date'].min().date()} — {d['Date'].max().date()}, "
          f"строк: {len(d)}, доля роста: {d['target'].mean():.3f}")


Удаляем из признаков (утечка): ['future_return_5d']

Корреляция признаков с будущей доходностью (должна быть близка к 0):
ret_1                -0.020
ret_5                -0.024
ret_20               -0.007
momentum_60          -0.005
volatility_5          0.053
volatility_20         0.059
ma_ratio_20          -0.019
ma_ratio_60          -0.018
volume_change         0.001
relative_volume_20    0.002
Name: future_return_5d, dtype: float64
Train      : 2015-04-01 — 2023-04-12, строк: 20220, доля роста: 0.565
Validation : 2023-04-20 — 2025-01-03, строк: 4300, доля роста: 0.570
Test       : 2025-01-14 — 2026-10-07, строк: 4350, доля роста: 0.566


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = "/content/drive/MyDrive/fintech_theme3/data"
os.makedirs(DATA_DIR, exist_ok=True)

train.to_csv(f"{DATA_DIR}/train.csv", index=False)
valid.to_csv(f"{DATA_DIR}/valid.csv", index=False)
test.to_csv(f"{DATA_DIR}/test.csv", index=False)
print("Сохранено в", DATA_DIR)

Mounted at /content/drive
Сохранено в /content/drive/MyDrive/fintech_theme3/data


In [ ]:
prices[["Date", "Ticker", "Close", "Volume"]].to_csv(f"{DATA_DIR}/prices.csv", index=False)
print("prices.csv сохранён")

prices.csv сохранён


Загружены дневные скорректированные котировки 10 акций из разных отраслей (AAPL, MSFT, AMZN, GOOGL, NVDA, JPM, KO, WMT, XOM, JNJ) за 2015-01-02 — 2026-10-14, по 2963 торговых дня на акцию, без пропусков и дубликатов. Целевая переменная — знак доходности на 5 торговых дней вперёд (shift(-5)); доля роста ~56–57% во всех выборках, классы примерно сбалансированы. Построено 10 признаков (доходности, волатильность, отклонение от скользящих средних, объём), все сдвинуты на 1 день (shift(1)) и используют только прошлые данные. Разбиение по датам 70/15/15 с зазором в 5 торговых дней между выборками, чтобы целевая переменная train не захватывала период validation. Столбец future_return_5d исключён из признаков как утечка; test не используется до этапа 4.